# 03 · Train the visual model on LRS3 trainval (GPU)
Add inputs: the outputs of **lrs3-trainval-0/1/2** and **lrs3-test**. Accelerator **GPU T4**, Internet **On**.

The training saves `last.pt` every epoch. If the 12-hour session ends, add this notebook's own previous output as input, set `RESUME_FROM`, and run again.

In [ ]:
# --- setup: clone the project code ---
import os, subprocess
if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/durgaanuhya02/MoVi-Swin-Motion--and-Viseme-Aware-Space-Time-Swin-Transformer-for-Robust-and-Efficient-Lip-Reading.git", "/kaggle/working/repo"], check=True)
os.chdir("/kaggle/working/repo")
!git log --oneline -1

In [ ]:
!pip install -q cmudict

In [ ]:
import glob, json
INDEX = sorted(glob.glob('/kaggle/input/**/lrs3_trainval_shard*/index.jsonl', recursive=True))
TEST_INDEX = glob.glob('/kaggle/input/**/lrs3_test_rois/index.jsonl', recursive=True)
INDEX_JSON = json.dumps(INDEX)
print(INDEX, TEST_INDEX)
RESUME_FROM = None   # e.g. '/kaggle/input/<previous-output>/runs/visual_lrs3'

### Check: do our trainval crops look like the pre-cropped LRS3 test crops?
Top rows: our MediaPipe crops (trainval). Bottom rows: test-set crops. If the mouth size or position differs a lot, re-run notebook 01 with a different `--crop-scale` before training.

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from src.data.dataset import ROIDataset
from src.preprocess.extract_rois import read_video
def frames_of(index, n):
    ds = ROIDataset(index)
    return [read_video(f"{e['_root']}/{e['roi']}")[10] for e in ds.entries[:n]]
row1 = frames_of(INDEX[0], 6); row2 = frames_of(TEST_INDEX[0], 6)
plt.figure(figsize=(12, 4)); plt.imshow(np.vstack([np.hstack(row1), np.hstack(row2)]), cmap='gray'); plt.axis('off')
plt.title('top: our trainval crops   |   bottom: LRS3 test crops'); plt.show()

In [ ]:
import shutil
OUT = '/kaggle/working/runs/visual_lrs3'
os.makedirs(OUT, exist_ok=True)
if RESUME_FROM:
    for f in ['last.pt', 'best.pt', 'log.jsonl']:
        if os.path.exists(f'{RESUME_FROM}/{f}'): shutil.copy(f'{RESUME_FROM}/{f}', OUT)

In [ ]:
!python -m src.train_visual --config configs/visual_lrs3.yaml --set 'index={INDEX_JSON}' out_dir={OUT}

In [ ]:
!tail -5 {OUT}/log.jsonl